# Similarity Visualization: OTP vs Smartcard Route Comparison

특정 OD의 스마트카드(관측) 통행과 OTP 후보 경로들의 유사도를 시각적으로 비교

In [ ]:
# -*- coding: utf-8 -*-
import sys, sqlite3, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import font_manager, rc

warnings.filterwarnings('ignore')

# 한글 폰트
font_path = 'C:/Windows/Fonts/malgun.ttf'
if Path(font_path).exists():
    font_manager.fontManager.addfont(font_path)
    rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path('../../').resolve()
DATA_DIR = ROOT / 'data'

sys.path.insert(0, str(Path('.').resolve()))
from module.similarity import (
    parse_smartcard_trip, compute_all_metrics, compute_composite_similarity,
    _normalize_stops_for_comparison, _lcs_length
)

In [ ]:
# ── 데이터 로드 ──
# OTP 캐시 (SQLite)
cache_db = DATA_DIR / 'training_set' / 'otp_cache.db'
conn = sqlite3.connect(str(cache_db))

# TCN 스마트카드 (하루치)
tcn_path = DATA_DIR / 'tcn' / '20250217' / 'TCN_20250217_route.parquet'
tcn = pd.read_parquet(tcn_path)

# 학습셋 (유사도 점수 포함)
training = pd.read_parquet(DATA_DIR / 'training_set' / 'route_choice_training.parquet')

print(f'TCN trips: {len(tcn):,}')
print(f'Training set: {len(training):,} rows')
print(f'Unique ODs in training: {training["od_pair"].nunique():,}')

In [ ]:
# ── 흥미로운 OD 탐색: 대안 4개 이상 + 선택 분산이 있는 OD ──
od_stats = training.groupby('od_pair').agg(
    n_alts=('alt_idx', 'count'),
    n_total=('n_total', 'first'),
    max_prob=('choice_prob', 'max'),
    min_prob=('choice_prob', 'min'),
).query('n_alts >= 4 and n_total >= 30')
od_stats['prob_spread'] = od_stats['max_prob'] - od_stats['min_prob']

# 선택 확률이 분산되어 있는 OD (min_prob > 0.05)
diverse_ods = od_stats[od_stats['min_prob'] > 0.05].sort_values('n_total', ascending=False)
print(f'다양한 선택 패턴을 보이는 OD: {len(diverse_ods)}개')
diverse_ods.head(10)

## OD 선택 및 유사도 계산

In [ ]:
# ── 분석할 OD 선택 (위 리스트에서 선택하거나 직접 입력) ──
# TARGET_OD = diverse_ods.index[0]  # 자동 선택
TARGET_OD = diverse_ods.index[0]  # 원하는 OD로 변경 가능

print(f'분석 대상 OD: {TARGET_OD}')

# Training set에서 해당 OD의 대안 정보
od_training = training[training['od_pair'] == TARGET_OD].sort_values('choice_prob', ascending=False)
print(f'대안 수: {len(od_training)}')
print()
for _, row in od_training.iterrows():
    print(f"  alt {int(row['alt_idx'])}: "
          f"prob={row['choice_prob']:.3f}, "
          f"composite={row['sim_composite']:.3f}, "
          f"mode={row['sim_mode']:.3f}, "
          f"route={row['sim_route']:.3f}, "
          f"seq={row['sim_sequence']:.3f}, "
          f"time={row['sim_time']:.3f}, "
          f"spatial={row['sim_spatial']:.3f}")

In [ ]:
# ── OTP 캐시에서 해당 OD의 경로 정보 로드 ──
cur = conn.cursor()
cur.execute('SELECT data FROM otp_cache WHERE od_pair = ?', (TARGET_OD,))
row = cur.fetchone()
if row is None:
    raise ValueError(f'OD {TARGET_OD} not found in OTP cache')

otp_data = pickle.loads(row[0])
otp_parsed_list = otp_data['otp_parsed']   # 각 대안의 파싱된 정보
alt_features = otp_data['alt_features']     # 각 대안의 경로 특성

print(f'OTP 대안 수: {len(otp_parsed_list)}')
for i, p in enumerate(otp_parsed_list):
    feat = alt_features[i]
    print(f"\n  Alt {i}: {feat['transport_category']}")
    print(f"    노선: {p['routes']}")
    print(f"    정류장: {p['stops']}")
    if p.get('full_stops'):
        print(f"    전체정류장({len(p['full_stops'])}): {p['full_stops'][:8]}{'...' if len(p['full_stops'])>8 else ''}")
    print(f"    소요시간: {p['total_time']//60}분 {p['total_time']%60}초")
    print(f"    환승: {p['transfer_count']}회")

In [ ]:
# ── TCN에서 해당 OD의 스마트카드 통행 샘플 로드 ──
sc_trips = tcn[tcn['od_pair'] == TARGET_OD]
print(f'TCN 통행 수 (20250217): {len(sc_trips)}')

# 대표 통행 패턴 추출 (노선명 기준 그룹화)
def trip_key(row):
    routes = row['노선명']
    if isinstance(routes, np.ndarray):
        routes = list(routes)
    return str(sorted(routes)) if routes is not None else ''

sc_trips = sc_trips.copy()
sc_trips['_key'] = sc_trips.apply(trip_key, axis=1)
pattern_counts = sc_trips['_key'].value_counts()
print(f'고유 통행 패턴: {len(pattern_counts)}개')
print()
for key, cnt in pattern_counts.items():
    sample = sc_trips[sc_trips['_key'] == key].iloc[0]
    print(f"  [{cnt}회] 노선={list(sample['노선명'])}, 정류장={list(sample['정류장명칭시퀀스'])}")

In [ ]:
# ── 대표 SC 통행 1개 선택 → 모든 OTP 대안과 유사도 계산 ──
# 가장 많은 패턴의 첫 번째 통행 사용
sc_sample = sc_trips[sc_trips['_key'] == pattern_counts.index[0]].iloc[0]
sc_parsed = parse_smartcard_trip(sc_sample)

print(f"SC 통행:")
print(f"  정류장: {sc_parsed['stops']}")
print(f"  노선: {sc_parsed['routes']}")
print(f"  모드: {sc_parsed['modes']}")
print(f"  환승: {sc_parsed['transfer_count']}회")
print(f"  소요시간: {sc_parsed['total_time']//60}분")
print()

# 각 OTP 대안과 유사도 계산
comparisons = []
for i, otp_p in enumerate(otp_parsed_list):
    metrics = compute_all_metrics(otp_p, sc_parsed)
    scores = compute_composite_similarity(metrics)
    comparisons.append({
        'alt_idx': i,
        'otp_parsed': otp_p,
        'metrics': metrics,
        'scores': scores,
    })
    print(f"  Alt {i} vs SC: composite={scores['composite']:.3f} "
          f"(mode={scores['mode_score']:.3f}, route={scores['route_score']:.3f}, "
          f"seq={scores['sequence_score']:.3f})")

## 시각화 1: 유사도 점수 비교 (레이더 차트 + 바 차트)

In [ ]:
# ── 유사도 점수 비교 바 차트 ──
n_alts = len(comparisons)
score_labels = ['Composite', 'Mode', 'Route', 'Sequence', 'Time', 'Spatial']
colors = plt.cm.Set2(np.linspace(0, 1, n_alts))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) 그룹 바 차트: 각 대안별 유사도 점수
ax = axes[0]
x = np.arange(len(score_labels))
width = 0.8 / n_alts

for i, comp in enumerate(comparisons):
    s = comp['scores']
    vals = [s['composite'], s['mode_score'], s['route_score'],
            s['sequence_score'], s['time_score'], s['spatial_score']]
    otp_p = comp['otp_parsed']
    label = f"Alt {i}: {','.join(otp_p['routes'])}"
    ax.bar(x + i * width, vals, width, label=label, color=colors[i], edgecolor='white')

ax.set_xticks(x + width * (n_alts - 1) / 2)
ax.set_xticklabels(score_labels)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title(f'OD: {TARGET_OD} — 유사도 점수 비교')
ax.legend(fontsize=8, loc='upper right')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# (2) 선택 확률 vs 유사도 composite
ax2 = axes[1]
composites = [c['scores']['composite'] for c in comparisons]
probs = []
for c in comparisons:
    match = od_training[od_training['alt_idx'] == c['alt_idx']]
    probs.append(match['choice_prob'].values[0] if len(match) > 0 else 0)

for i in range(n_alts):
    otp_p = comparisons[i]['otp_parsed']
    label = f"Alt {i}: {','.join(otp_p['routes'])}"
    ax2.scatter(composites[i], probs[i], s=200, color=colors[i],
                edgecolors='black', zorder=5, label=label)
    ax2.annotate(f'Alt {i}', (composites[i], probs[i]),
                 textcoords='offset points', xytext=(8, 5), fontsize=9)

ax2.set_xlabel('Composite Similarity')
ax2.set_ylabel('Choice Probability')
ax2.set_title('유사도 vs 선택 확률')
ax2.set_xlim(0, 1.05)
ax2.set_ylim(-0.02, 1.05)
ax2.legend(fontsize=8)
for spine in ['top', 'right']:
    ax2.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(str(DATA_DIR / 'similarity' / f'sim_scores_{TARGET_OD}.png'), dpi=150, bbox_inches='tight')
plt.show()

## 시각화 2: 정류장 시퀀스 비교 (Alignment Diagram)

SC 통행의 정류장 시퀀스와 각 OTP 대안의 정류장 시퀀스를 나란히 비교.
- 초록: 일치하는 정류장 (LCS 기준)
- 빨강: 불일치 정류장

In [ ]:
def get_lcs_indices(seq1, seq2):
    """LCS에 포함되는 인덱스 쌍을 역추적으로 구함"""
    m, n = len(seq1), len(seq2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[i-1] == seq2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    # 역추적
    idx1, idx2 = [], []
    i, j = m, n
    while i > 0 and j > 0:
        if seq1[i-1] == seq2[j-1]:
            idx1.append(i-1)
            idx2.append(j-1)
            i -= 1
            j -= 1
        elif dp[i-1][j] > dp[i][j-1]:
            i -= 1
        else:
            j -= 1
    return list(reversed(idx1)), list(reversed(idx2))


def draw_sequence_alignment(sc_stops, otp_stops_list, otp_labels, title, save_path=None):
    """
    SC 정류장 시퀀스 vs 여러 OTP 대안의 정류장 시퀀스를 시각적으로 비교.
    각 대안별로 행을 나누고, 일치/불일치 정류장을 색상으로 표시.
    """
    n_alts = len(otp_stops_list)
    fig_height = max(4, 1.5 * (n_alts + 1) + 1)
    
    # 최대 정류장 수 (x축 범위)
    all_lens = [len(sc_stops)] + [len(s) for s in otp_stops_list]
    max_stops = max(all_lens)
    fig_width = max(14, max_stops * 1.8)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    
    y_positions = list(range(n_alts + 1, 0, -1))  # SC가 맨 위
    row_height = 0.35
    
    # SC 행 (맨 위)
    y_sc = y_positions[0]
    for j, stop in enumerate(sc_stops):
        ax.add_patch(plt.Rectangle((j - 0.4, y_sc - row_height),
                                    0.8, row_height * 2,
                                    facecolor='#4CAF50', edgecolor='white',
                                    linewidth=1.5, alpha=0.85))
        ax.text(j, y_sc, stop, ha='center', va='center',
                fontsize=8, fontweight='bold', color='white')
    ax.text(-1.5, y_sc, 'SC (관측)', ha='right', va='center',
            fontsize=10, fontweight='bold', color='#333')
    
    # 각 OTP 대안 행
    for alt_i in range(n_alts):
        y_otp = y_positions[alt_i + 1]
        otp_stops = otp_stops_list[alt_i]
        
        # SC와의 LCS 계산 (정규화 후)
        norm_otp, norm_sc = _normalize_stops_for_comparison(otp_stops, sc_stops)
        lcs_idx_otp, lcs_idx_sc = get_lcs_indices(norm_otp, norm_sc)
        lcs_set_otp = set(lcs_idx_otp)
        
        for j, stop in enumerate(otp_stops):
            is_match = j in lcs_set_otp
            color = '#4CAF50' if is_match else '#F44336'
            alpha = 0.85 if is_match else 0.6
            
            ax.add_patch(plt.Rectangle((j - 0.4, y_otp - row_height),
                                        0.8, row_height * 2,
                                        facecolor=color, edgecolor='white',
                                        linewidth=1.5, alpha=alpha))
            ax.text(j, y_otp, stop, ha='center', va='center',
                    fontsize=8, color='white')
        
        # 매칭 연결선 (SC ↔ OTP)
        for oi, si in zip(lcs_idx_otp, lcs_idx_sc):
            ax.plot([si, oi], [y_sc - row_height - 0.05, y_otp + row_height + 0.05],
                    color='#4CAF50', alpha=0.3, linewidth=1.5, linestyle='--')
        
        # LCS 점수
        lcs_score = len(lcs_idx_otp) / max(len(norm_otp), len(norm_sc)) if max(len(norm_otp), len(norm_sc)) > 0 else 0
        label = f'{otp_labels[alt_i]}  (LCS={lcs_score:.2f})'
        ax.text(-1.5, y_otp, label, ha='right', va='center',
                fontsize=9, color='#555')
    
    # 범례
    legend_elements = [
        mpatches.Patch(facecolor='#4CAF50', alpha=0.85, label='일치 (LCS)'),
        mpatches.Patch(facecolor='#F44336', alpha=0.6, label='불일치'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    
    ax.set_xlim(-2, max_stops + 0.5)
    ax.set_ylim(0.2, n_alts + 2.2)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
    ax.axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 정류장 시퀀스 비교 그리기 ──
sc_stops = sc_parsed['stops']
otp_stops_list = [c['otp_parsed']['stops'] for c in comparisons]
otp_labels = [f"Alt {c['alt_idx']}: {','.join(c['otp_parsed']['routes'])}" for c in comparisons]

draw_sequence_alignment(
    sc_stops, otp_stops_list, otp_labels,
    title=f'OD: {TARGET_OD} — 정류장 시퀀스 비교 (boarding/alighting)',
    save_path=str(DATA_DIR / 'similarity' / f'sim_sequence_{TARGET_OD}.png')
)

## 시각화 3: 세부 유사도 지표 히트맵

각 대안 × 세부 지표를 히트맵으로 한눈에 비교

In [ ]:
# ── 세부 지표 히트맵 ──
metric_names = [
    'mode_exact', 'mode_jaccard',
    'route_exact', 'route_jaccard', 'route_main',
    'seq_jaccard', 'seq_lcs', 'seq_levenshtein', 'seq_prefix', 'seq_suffix',
    'time_ratio', 'time_band', 'time_score',
    'polyline_similarity',
]
metric_labels = [
    'Mode\nExact', 'Mode\nJaccard',
    'Route\nExact', 'Route\nJaccard', 'Route\nMain',
    'Seq\nJaccard', 'Seq\nLCS', 'Seq\nLevenshtein', 'Seq\nPrefix', 'Seq\nSuffix',
    'Time\nRatio', 'Time\nBand', 'Time\nScore',
    'Spatial\nSim',
]

# 카테고리 구분 위치
cat_boundaries = [0, 2, 5, 10, 13, 14]
cat_names = ['Mode', 'Route', 'Sequence', 'Time', 'Spatial']
cat_colors = ['#E3F2FD', '#FFF3E0', '#E8F5E9', '#FCE4EC', '#F3E5F5']

data = np.zeros((n_alts, len(metric_names)))
row_labels = []
for i, comp in enumerate(comparisons):
    for j, mn in enumerate(metric_names):
        data[i, j] = comp['metrics'].get(mn, 0)
    row_labels.append(f"Alt {i}: {','.join(comp['otp_parsed']['routes'])}")

fig, ax = plt.subplots(figsize=(18, max(3, n_alts * 0.8 + 2)))
im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

# 셀 값 표시
for i in range(n_alts):
    for j in range(len(metric_names)):
        val = data[i, j]
        color = 'white' if val < 0.3 or val > 0.85 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color=color, fontweight='bold')

# 카테고리 구분 배경
for ci in range(len(cat_names)):
    x_start = cat_boundaries[ci] - 0.5
    x_end = cat_boundaries[ci + 1] - 0.5
    ax.axvline(x=x_end, color='white', linewidth=2)
    ax.text((x_start + x_end) / 2, -1.2, cat_names[ci],
            ha='center', va='center', fontsize=10, fontweight='bold',
            color='#333',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=cat_colors[ci], edgecolor='none'))

ax.set_xticks(range(len(metric_labels)))
ax.set_xticklabels(metric_labels, fontsize=7, rotation=0)
ax.set_yticks(range(n_alts))
ax.set_yticklabels(row_labels, fontsize=9)
ax.set_title(f'OD: {TARGET_OD} — 세부 유사도 지표', fontsize=13, fontweight='bold', pad=30)

cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Score', fontsize=10)

plt.tight_layout()
plt.savefig(str(DATA_DIR / 'similarity' / f'sim_heatmap_{TARGET_OD}.png'), dpi=150, bbox_inches='tight')
plt.show()

## 시각화 4: 경로 구조 종합 비교 (Summary Card)

SC 통행 vs 각 OTP 대안을 카드 형태로 종합 비교 — 노선, 모드, 환승, 시간, 정류장

In [ ]:
# ── 경로 구조 종합 비교표 ──
def draw_route_summary(sc_parsed, comparisons, od_training, target_od, save_path=None):
    n_alts = len(comparisons)
    n_cols = n_alts + 1  # SC + OTP alts
    
    fig, ax = plt.subplots(figsize=(max(12, n_cols * 3.5), 8))
    ax.axis('off')
    
    # 행 정보
    row_labels = ['', '노선', '모드', '환승', '소요시간', '정류장',
                  '', 'Composite', 'Mode', 'Route', 'Sequence', 'Time', 'Spatial',
                  '', '선택확률']
    n_rows = len(row_labels)
    
    col_width = 1.0 / n_cols
    row_height = 1.0 / n_rows
    
    # 헤더 행
    headers = ['SC (관측)'] + [f"Alt {c['alt_idx']}" for c in comparisons]
    for ci, header in enumerate(headers):
        color = '#4CAF50' if ci == 0 else '#2196F3'
        ax.add_patch(plt.Rectangle((ci * col_width, 1 - row_height),
                                    col_width, row_height,
                                    facecolor=color, edgecolor='white', linewidth=2))
        ax.text(ci * col_width + col_width/2, 1 - row_height/2, header,
                ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    
    # SC 열 데이터
    sc_data = [
        '',
        ', '.join(sc_parsed['routes']),
        ', '.join(sorted(sc_parsed['modes'])),
        f"{sc_parsed['transfer_count']}회",
        f"{sc_parsed['total_time']//60}분",
        ' → '.join(sc_parsed['stops']),
        '', '', '', '', '', '', '', '', ''
    ]
    
    # 각 OTP 열 데이터
    alt_data_list = []
    for comp in comparisons:
        otp_p = comp['otp_parsed']
        s = comp['scores']
        match = od_training[od_training['alt_idx'] == comp['alt_idx']]
        prob = match['choice_prob'].values[0] if len(match) > 0 else 0
        
        alt_data_list.append([
            '',
            ', '.join(otp_p['routes']),
            ', '.join(sorted(otp_p['modes'])),
            f"{otp_p['transfer_count']}회",
            f"{otp_p['total_time']//60}분",
            ' → '.join(otp_p['stops']),
            '',
            f"{s['composite']:.3f}",
            f"{s['mode_score']:.3f}",
            f"{s['route_score']:.3f}",
            f"{s['sequence_score']:.3f}",
            f"{s['time_score']:.3f}",
            f"{s['spatial_score']:.3f}",
            '',
            f"{prob:.1%}",
        ])
    
    # 섹션 헤더 행
    section_rows = {0: '경로 정보', 6: '유사도 점수', 13: '선택'}
    
    for ri in range(1, n_rows):
        y_top = 1 - (ri + 1) * row_height
        
        # 섹션 헤더
        if ri in section_rows:
            ax.add_patch(plt.Rectangle((0, y_top), 1.0, row_height,
                                        facecolor='#ECEFF1', edgecolor='white', linewidth=1))
            ax.text(0.5, y_top + row_height/2, section_rows[ri],
                    ha='center', va='center', fontsize=10, fontweight='bold', color='#555')
            continue
        
        # SC 열
        bg_sc = '#E8F5E9' if ri % 2 == 0 else '#F1F8E9'
        ax.add_patch(plt.Rectangle((0, y_top), col_width, row_height,
                                    facecolor=bg_sc, edgecolor='white', linewidth=1))
        ax.text(col_width/2, y_top + row_height/2, sc_data[ri],
                ha='center', va='center', fontsize=8, color='#333')
        
        # OTP 열들
        for ci, alt_data in enumerate(alt_data_list):
            bg = '#E3F2FD' if ri % 2 == 0 else '#BBDEFB'
            # 유사도 점수 색상 강조
            if 7 <= ri <= 12 and alt_data[ri]:
                val = float(alt_data[ri])
                if val >= 0.8:
                    bg = '#C8E6C9'
                elif val >= 0.5:
                    bg = '#FFF9C4'
                else:
                    bg = '#FFCDD2'
            
            x_pos = (ci + 1) * col_width
            ax.add_patch(plt.Rectangle((x_pos, y_top), col_width, row_height,
                                        facecolor=bg, edgecolor='white', linewidth=1))
            ax.text(x_pos + col_width/2, y_top + row_height/2, alt_data[ri],
                    ha='center', va='center', fontsize=8, color='#333')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f'OD: {target_od} — 경로 구조 종합 비교', fontsize=13, fontweight='bold', pad=10)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

draw_route_summary(sc_parsed, comparisons, od_training, TARGET_OD,
                   save_path=str(DATA_DIR / 'similarity' / f'sim_summary_{TARGET_OD}.png'))

## 여러 OD 일괄 비교

위 리스트에서 다양한 OD를 선택하여 일괄 시각화

In [ ]:
# ── 여러 OD 일괄 시각화 ──
# diverse_ods에서 상위 N개 OD 선택
N_ODS = 3  # 원하는 만큼 변경

for od_idx in range(min(N_ODS, len(diverse_ods))):
    od_pair = diverse_ods.index[od_idx]
    print(f'\n{"="*60}')
    print(f'OD: {od_pair}')
    print(f'{"="*60}')
    
    # OTP 캐시 로드
    cur.execute('SELECT data FROM otp_cache WHERE od_pair = ?', (od_pair,))
    row = cur.fetchone()
    if row is None:
        print(f'  OTP 캐시 없음, 건너뜀')
        continue
    
    otp_data = pickle.loads(row[0])
    otp_parsed = otp_data['otp_parsed']
    
    # TCN 통행
    sc_od = tcn[tcn['od_pair'] == od_pair]
    if len(sc_od) == 0:
        print(f'  TCN 통행 없음 (20250217), 건너뜀')
        continue
    
    sc_p = parse_smartcard_trip(sc_od.iloc[0])
    od_tr = training[training['od_pair'] == od_pair].sort_values('choice_prob', ascending=False)
    
    # 유사도 계산
    comps = []
    for i, otp_p in enumerate(otp_parsed):
        metrics = compute_all_metrics(otp_p, sc_p)
        scores = compute_composite_similarity(metrics)
        comps.append({'alt_idx': i, 'otp_parsed': otp_p, 'metrics': metrics, 'scores': scores})
    
    # 시퀀스 비교
    otp_stops = [c['otp_parsed']['stops'] for c in comps]
    otp_lbls = [f"Alt {c['alt_idx']}: {','.join(c['otp_parsed']['routes'])}" for c in comps]
    
    draw_sequence_alignment(
        sc_p['stops'], otp_stops, otp_lbls,
        title=f'OD: {od_pair} — 정류장 시퀀스 비교',
        save_path=str(DATA_DIR / 'similarity' / f'sim_sequence_{od_pair}.png')
    )
    
    # 종합 비교표
    draw_route_summary(sc_p, comps, od_tr, od_pair,
                       save_path=str(DATA_DIR / 'similarity' / f'sim_summary_{od_pair}.png'))

In [ ]:
# ── 정리 ──
conn.close()
print('Done!')